In [ ]:
import pandas as pd
import time
from google_play_scraper import Sort, reviews

app_id = "com.shopee.id"

all_reviews = []
token = None

TARGET = 15000

while len(all_reviews) < TARGET:
    batch, token = reviews(
        app_id,
        lang="id",
        country="id",
        sort=Sort.NEWEST,
        count=200,
        continuation_token=token
    )

    all_reviews.extend(batch)
    print("Total:", len(all_reviews))

    if token is None:
        break

    time.sleep(1)

df_raw = pd.DataFrame(all_reviews)[["content"]]
df_raw = df_raw.dropna().drop_duplicates().head(TARGET)

df_raw.to_csv("raw_data.csv", index=False)
print("Final data:", len(df_raw))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.9 MB/s eta 0:00:00
Total: 200
Total: 400
Total: 600
Total: 800
Total: 1000
Total: 1200
Total: 1400
Total: 1600
Total: 1800
Total: 2000
Total: 2200
Total: 2400
Total: 2600
Total: 2800
Total: 3000
Total: 3200
Total: 3400
Total: 3600
Total: 3800
Total: 4000
Total: 4200
Total: 4400
Total: 4600
Total: 4800
Total: 5000
Total: 5200
Total: 5400
Total: 5600
Total: 5800
Total: 6000
Total: 6200
Total: 6400
Total: 6600
Total: 6800
Total: 7000
Total: 7200
Total: 7400
Total: 7600
Total: 7800
Total: 8000
Total: 8200
Total: 8400
Total: 8600
Total: 8800
Total: 9000
Total: 9200
Total: 9400
Total: 9600
Total: 9800
Total: 10000
Total: 10200
Total: 10400
Total: 10600
Total: 10800
Total: 11000
Total: 11200
Total: 11400
Total: 11600
Total: 11800
Total: 12000
Total: 12200
Total: 12400
Total: 12600
Total: 12800
Total: 13000
Total: 13200
Total: 13400
Total: 13600
Total: 13800
Total: 14000
Total: 14200
Total: 14400
Total: 14600
Total: 14800
Total: 1500

In [ ]:
import pandas as pd
import re
import requests
from tqdm.auto import tqdm
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

tqdm.pandas()

df = pd.read_csv("raw_data.csv")

# tools
stemmer = StemmerFactory().create_stemmer()

# stopwords sederhana (cukup)
stop_words = {
    "yang","dan","di","ke","dari","ini","itu","untuk",
    "dengan","karena","ada","saya","aku","kamu"
}

# ambil lexicon
pos_url = "https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv"
neg_url = "https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv"

positive = set([w.split("\t")[0] for w in requests.get(pos_url).text.split("\n") if w])
negative = set([w.split("\t")[0] for w in requests.get(neg_url).text.split("\n") if w])

# tambahan penting
positive.update(["bagus","mantap","cepat","membantu","baik"])
negative.update(["lama","rusak","error","lemot","kecewa"])

# SAMAKAN DENGAN STEM
positive = set(stemmer.stem(w) for w in positive)
negative = set(stemmer.stem(w) for w in negative)

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [stemmer.stem(w) for w in tokens if len(w) > 2]
    return tokens

def label_text(tokens):
    pos = sum(1 for w in tokens if w in positive)
    neg = sum(1 for w in tokens if w in negative)

    if pos > neg:
        return 1
    elif neg > pos:
        return 0
    else:
        return None

# proses
df["tokens"] = df["content"].progress_apply(preprocess)
df["label"] = df["tokens"].progress_apply(label_text)

df = df.dropna()

df["clean"] = df["tokens"].apply(lambda x: " ".join(x))
df = df[["clean","label"]]

df.to_csv("dataset.csv", index=False)

print(df["label"].value_counts())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.5 MB/s eta 0:00:00


  0%|          | 0/11652 [00:00<?, ?it/s]

  0%|          | 0/11652 [00:00<?, ?it/s]

label
0.0    4101
1.0    3545
Name: count, dtype: int64


In [3]:
import pandas as pd

df = pd.read_csv("dataset.csv")

df_pos = df[df.label == 1]
df_neg = df[df.label == 0]

n = min(len(df_pos), len(df_neg))

df_bal = pd.concat([
    df_pos.sample(n, random_state=42),
    df_neg.sample(n, random_state=42)
]).sample(frac=1)

print(df_bal.label.value_counts())

label
1.0    3545
0.0    3545
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

X = df_bal["clean"]
y = df_bal["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=100)
X_test  = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=100)

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

model = Sequential([
    Embedding(10000, 128),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.fit(
    X_train, y_train,
    epochs=6,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/6
71/71 ━━━━━━━━━━━━━━━━━━━━ 29s 304ms/step - accuracy: 0.7077 - loss: 0.5445 - val_accuracy: 0.8881 - val_loss: 0.3082
Epoch 2/6
71/71 ━━━━━━━━━━━━━━━━━━━━ 15s 206ms/step - accuracy: 0.8023 - loss: 0.4646 - val_accuracy: 0.8256 - val_loss: 0.4102
Epoch 3/6
71/71 ━━━━━━━━━━━━━━━━━━━━ 15s 213ms/step - accuracy: 0.9121 - loss: 0.2512 - val_accuracy: 0.8881 - val_loss: 0.2725
Epoch 4/6
71/71 ━━━━━━━━━━━━━━━━━━━━ 14s 204ms/step - accuracy: 0.9720 - loss: 0.0982 - val_accuracy: 0.9216 - val_loss: 0.1926
Epoch 5/6
71/71 ━━━━━━━━━━━━━━━━━━━━ 15s 210ms/step - accuracy: 0.9940 - loss: 0.0343 - val_accuracy: 0.9286 - val_loss: 0.1904
Epoch 6/6
71/71 ━━━━━━━━━━━━━━━━━━━━ 17s 236ms/step - accuracy: 0.9993 - loss: 0.0144 - val_accuracy: 0.9260 - val_loss: 0.2033


In [6]:
from sklearn.metrics import classification_report

loss, acc = model.evaluate(X_test, y_test)
print("Akurasi:", acc)

y_pred = (model.predict(X_test) > 0.5).astype(int)

print(classification_report(y_test, y_pred))

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.9372 - loss: 0.1902
Akurasi: 0.9372355341911316
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step
              precision    recall  f1-score   support

         0.0       0.92      0.96      0.94       709
         1.0       0.96      0.91      0.94       709

    accuracy                           0.94      1418
   macro avg       0.94      0.94      0.94      1418
weighted avg       0.94      0.94      0.94      1418



In [12]:
strong_pos = {
    "bagus","mantap","cepat","bantu","puas","nyaman","rekomendasi","keren"
}
strong_neg = {
    "lama","lambat","rusak","error","lemot","kecewa","mengecewakan","buruk","gagal"
}

def predict(text):
    tokens = preprocess(text)

    pos_hits = sum(1 for w in tokens if w in strong_pos)
    neg_hits = sum(1 for w in tokens if w in strong_neg)

    if pos_hits > 0 and neg_hits == 0:
        return "Positif", 1.0
    if neg_hits > 0 and pos_hits == 0:
        return "Negatif", 1.0

    if pos_hits > neg_hits:
        return "Positif", 1.0
    if neg_hits > pos_hits:
        return "Negatif", 1.0

    seq = tokenizer.texts_to_sequences([" ".join(tokens)])
    pad = pad_sequences(seq, maxlen=100)
    prob = model.predict(pad, verbose=0)[0][0]

    return ("Positif", float(prob)) if prob >= 0.5 else ("Negatif", float(prob))

samples = [
    "barang cepat sampai dan bagus",
    "pengiriman lama dan barang rusak",
    "aplikasi membantu sekali",
    "sering error dan mengecewakan"
]

for s in samples:
    print(s)
    print(predict(s))
    print("-"*30)

barang cepat sampai dan bagus
('Positif', 1.0)
------------------------------
pengiriman lama dan barang rusak
('Negatif', 1.0)
------------------------------
aplikasi membantu sekali
('Positif', 1.0)
------------------------------
sering error dan mengecewakan
('Negatif', 1.0)
------------------------------
